# Exploration

Quick, lightweight look at the generated and cleaned data plus the pipeline
outputs. This notebook does not need Spark: it reads a sample of the Parquet
data with PyArrow and the small result files written by the scripts, so it runs
in the local virtualenv on any OS.

The scripts in `src/` are the source of truth; run them first (see the README).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import pyarrow.dataset as ds

# Work from the project root whether Jupyter starts in notebooks/ or the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import config  # noqa: E402

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Raw data (Stage 1)

In [ ]:
raw = ds.dataset(str(config.RAW_DIR), format="parquet", partitioning="hive")
files = list(config.RAW_DIR.rglob("*.parquet"))
size_gb = sum(f.stat().st_size for f in files) / 1024 ** 3
print(f"{raw.count_rows():,} rows | {len(files):,} files | {size_gb:.2f} GB on disk")
raw.schema

In [ ]:
sample = raw.head(100_000).to_pandas()
sample.head()

In [ ]:
# Data-quality issues are visible even in a small sample
pd.Series({
    "missing email %": sample["email"].isna().mean() * 100,
    "missing city %": sample["city"].isna().mean() * 100,
    "negative price %": (sample["price"] <= 0).mean() * 100,
    "negative quantity %": (sample["quantity"] <= 0).mean() * 100,
    "total_amount mismatch %": ((sample["price"] * sample["quantity"]
                                 - sample["total_amount"]).abs() > 0.01).mean() * 100,
}).round(2)

In [ ]:
sample[["price", "quantity", "total_amount"]].describe().round(2)

In [ ]:
sample["payment_method"].value_counts(normalize=True).round(3)

## 2. Quality report and cleaning (Stages 3-5, full dataset)

In [ ]:
print((config.REPORTS_DIR / "exploration_output.txt").read_text()[:1200])

In [ ]:
pd.read_csv(config.REPORTS_DIR / "data_quality_report.csv")

In [ ]:
pd.read_csv(config.REPORTS_DIR / "cleaning_summary.csv")

## 3. Analytics results (Stage 6)

In [ ]:
results = config.OUTPUTS_DIR / "query_results"
sorted(p.stem for p in results.glob("*.csv"))

In [ ]:
pd.read_csv(results / "revenue_by_category.csv")

In [ ]:
monthly = pd.read_csv(results / "monthly_revenue_trend.csv").sort_values("year_month")
monthly.plot(x="year_month", y="monthly_revenue", figsize=(12, 4), legend=False,
             title="Monthly revenue");

## 4. Performance (Stages 2 and 7)

In [ ]:
pd.read_csv(config.REPORTS_DIR / "pandas_test_results.csv")

In [ ]:
pd.read_csv(config.REPORTS_DIR / "performance_results.csv")